# ChemistryInstructor — Unsloth Fine-Tune (Kaggle)

**NPC:** ChemistryInstructor  
**Subject:** General chemistry: atoms, molecules, elements, the periodic table, chemical reactions, stoichiometry, acids and bases  
**Model:** unsloth/Llama-3.1-8B-Instruct-bnb-4bit  
**Key:** chemistry_instructor

---
### Supabase Database
The project uses a local Supabase backend for NPC session state, player/NPC memory, semantic embeddings, and relationship analytics.

Local Supabase example endpoints:
- PostgreSQL: `localhost:15433`
- Studio: `localhost:16434`
- Kong/API gateway: `localhost:16433`

Database schema highlights:
- `dialogue_sessions` – session metadata for player/NPC interactions
- `dialogue_turns` – text turns for every conversation step
- `npc_memories` – generated summaries for recall and persistence
- `player_profiles` – player identity and metadata
- `npc_profiles` – NPC configuration, personality, voice rules, and domain knowledge
- `player_memory_embeddings` – vector embeddings for semantic memory retrieval
- `dialogue_turn_embeddings` – embeddings for conversation turns
- `relation_graph_nodes` / `relation_graph_edges` – graph-based relationship storage
- `dialogue_relation_terms` – search terms used to derive relation graphs
- `test_results` – stores evaluation/test metrics and run details

Key stored procedures:
- `get_or_create_session()`
- `insert_turn_fast()`
- `summarize_dialogue_session()`
- `get_player_npc_memory()`
- `get_god_memory()`
- `search_memories_semantic()`
- `generate_dialogue_relation_graph()`
- `get_dialogue_relation_matches()`
- `upsert_npc_profile()` / `get_npc_profile()`

Use Supabase for Unity backend APIs that need:
- session lifecycle and turn history
- memory persistence and semantic retrieval
- NPC catalog and profile lookup
- relationship/graph analytics
- evaluation/test result tracking

---
### Setup
1. **Settings → Accelerator** → **GPU P100** or **GPU T4 x2**
2. **Settings → Internet** → **On**
3. Dataset is pre-attached: `andreathar/chemistry-instructor-npc` → `/kaggle/input/chemistry-instructor-npc/chemistry_instructor.jsonl`
4. **Run all cells**

This notebook will:
1. Install Unsloth and dependencies
2. Load unsloth/Llama-3.1-8B-Instruct-bnb-4bit in 4-bit
3. Apply LoRA adapters
4. Load the training dataset from `/kaggle/input/`
5. Run SFT training
6. Export LoRA adapter to GGUF format (small file for Unity runtime loading)
7. Save GGUF to `/kaggle/working/` for download from the Output tab

In [ ]:
%%capture
import subprocess, sys, os

TORCH_PREFIX = '/kaggle/working/torch_env'
os.makedirs(TORCH_PREFIX, exist_ok=True)

# Install cu121 torch (supports P100 sm_60 + T4 sm_75)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--target', TORCH_PREFIX, '--upgrade',
    'torch==2.3.1+cu121', 'torchvision==0.18.1+cu121', 'torchaudio==2.3.1+cu121',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)

# Install unsloth + compatible torchao to same prefix
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--target', TORCH_PREFIX,
    'unsloth', 'torchao==0.6.1', 'gguf', 'huggingface_hub', 'datasets'
], check=True)

# Set PYTHONPATH so all subsequent cells pick up our torch first
os.environ['PYTHONPATH'] = TORCH_PREFIX + ':' + os.environ.get('PYTHONPATH', '')
# Also inject into current sys.path for this cell
if TORCH_PREFIX not in sys.path:
    sys.path.insert(0, TORCH_PREFIX)

print('Install done. PYTHONPATH set.')


## Unsloth
Load the base model with 4-bit quantization.

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path:
    sys.path.insert(0, TORCH_PREFIX)

import torch
print(f'torch={torch.__version__}, CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Smoke test
    x = torch.ones(2).cuda()
    print(f'CUDA smoke test: {x.sum().item()} OK')

from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print('Model loaded.')


In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = max_seq_length,
)

## Data Preparation
Apply ChatML template and load the training dataset from the attached Kaggle dataset.

Dataset is pre-attached as `andreathar/chemistry-instructor-npc`. Expected path:
`/kaggle/input/chemistry-instructor-npc/chemistry_instructor.jsonl`

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

import os
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

# Dataset path — attached as andreathar/chemistry-instructor-npc
# Slug: chemistry-instructor-npc  →  /kaggle/input/chemistry-instructor-npc/
DATASET_PATH = "/kaggle/input/chemistry-instructor-npc/chemistry_instructor.jsonl"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}.\n"
        "Please attach the 'chemistry-instructor-npc' Kaggle dataset to this notebook "
        "via Settings → Add Data before running."
    )

print(f"Loading dataset: {DATASET_PATH}")
dataset = load_dataset(
    "json",
    data_files = DATASET_PATH,
    split = "train",
)
print(f"Loaded {len(dataset)} training examples")

def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
    )
    return example

dataset = dataset.map(format_chat)
print(f"Example text:\n{dataset[0]['text'][:200]}...")

## Training
Configure and run SFT training with Unsloth.

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

import torch

per_device_train_batch_size = 2
gradient_accumulation_steps = 4
max_steps = -1
num_train_epochs = 3
learning_rate = 0.0002
warmup_steps = 10
weight_decay = 0.01
optim = "adamw_8bit"
lr_scheduler_type = "linear"
seed = 42
output_dir = "/kaggle/working/kaggle_output"

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

from trl import SFTTrainer
from transformers import TrainingArguments

# packing is disabled: train_on_responses_only (applied below) requires
# packing=False to correctly mask instruction tokens at sequence boundaries.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        output_dir = output_dir,
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        num_train_epochs = num_train_epochs,
        learning_rate = learning_rate,
        warmup_steps = warmup_steps,
        weight_decay = weight_decay,
        logging_steps = 1,
        optim = optim,
        lr_scheduler_type = lr_scheduler_type,
        seed = seed,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        report_to = "none",
        remove_unused_columns = False,
        ddp_find_unused_parameters = False if torch.cuda.device_count() > 1 else None,
        dataloader_pin_memory = False,
    ),
)

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
    tokenizer = tokenizer,
)

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

# Verify label masking works (check what % of tokens contribute to loss)
# This manually masks instruction tokens to avoid depending on internal APIs.
for i, example in enumerate(dataset.select(range(2))):
    tokenized = tokenizer(example["text"], truncation=True, max_length=max_seq_length, return_tensors="pt")
    input_ids = tokenized.input_ids[0].clone()
    labels = input_ids.clone()
    # Mask everything before "assistant" (the instruction/user part)
    text = example["text"]
    marker = "<|im_start|>assistant"
    if marker in text:
        pre = tokenizer(text[:text.index(marker)], truncation=True, max_length=max_seq_length)
        labels[:len(pre.input_ids)] = -100
    non_zero = (labels != -100).sum().item()
    total = labels.numel()
    print(f"Example {i}: {non_zero}/{total} tokens contribute to loss ({100 * non_zero / total:.1f}%)")

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

import time
train_start = time.time()

trainer_stats = trainer.train()

train_elapsed = time.time() - train_start
print(f"\nTraining completed in {train_elapsed / 60:.1f} minutes")

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Peak reserved memory = {used_memory} GB")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB")
print(f"Peak reserved memory % of max = {used_percentage} %")
print(f"Peak reserved memory for training % of max = {lora_percentage} %")

## LoRA Adapter Export to GGUF
Save the trained LoRA adapter and convert to GGUF format for Unity runtime loading.
This produces a **small** file (~10-30 MB) — not a full model merge.

The output GGUF will be saved to `/kaggle/working/` and available for download from the **Output** tab.

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

import os, sys, subprocess, shutil, json, tempfile
from pathlib import Path

# 1. Save the PEFT LoRA adapter (just delta weights, ~10-30 MB)
adapter_dir = "/kaggle/working/outputs/chemistry_instructor"
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)  # needed by convert_lora_to_gguf.py
adapter_files = [f for f in os.listdir(adapter_dir) if os.path.isfile(os.path.join(adapter_dir, f))]
total_mb = sum(os.path.getsize(os.path.join(adapter_dir, f)) for f in adapter_files) / (1024 * 1024)
print(f"LoRA adapter saved to {adapter_dir} ({total_mb:.1f} MB)")
for f in adapter_files:
    size_mb = os.path.getsize(os.path.join(adapter_dir, f)) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

# 2. Find the llama.cpp converter (installed by Unsloth during first GGUF export)
print("\nLooking for convert_lora_to_gguf.py...")
converter = None
# Kaggle working dir and common pip install locations
for search_root in ["/kaggle/working", "/usr", "/opt", "/root"]:
    if os.path.isdir(search_root):
        for root, dirs, files in os.walk(search_root):
            if "convert_lora_to_gguf.py" in files:
                converter = os.path.join(root, "convert_lora_to_gguf.py")
                break
        if converter:
            break

if not converter:
    # Fallback: clone llama.cpp into /kaggle/working/
    print("Downloading llama.cpp to get converter...")
    os.system("git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp 2>/dev/null")
    converter = "/kaggle/working/llama.cpp/convert_lora_to_gguf.py"

if os.path.exists(converter):
    out_file = "/kaggle/working/chemistry_instructor-lora.f16.gguf"
    print(f"Converting LoRA to GGUF using: {converter}")
    print(f"Output: {out_file}")

    # Get clean base model config (strip bitsandbytes quantization keys)
    adapter_config_path = os.path.join(adapter_dir, "adapter_config.json")
    base_model = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"  # fallback
    if os.path.exists(adapter_config_path):
        with open(adapter_config_path) as f:
            ac = json.load(f)
        base_model = ac.get("base_model_name_or_path", base_model)
        print(f"Base model from adapter config: {base_model}")

    # Fetch clean config.json from HuggingFace hub
    tmp_config_dir = tempfile.mkdtemp(prefix="lora-gguf-config-")
    try:
        from huggingface_hub import hf_hub_download
        config_path = hf_hub_download(base_model, "config.json")
        shutil.copy2(config_path, os.path.join(tmp_config_dir, "config.json"))
        print(f"Using config from HF cache: {config_path}")
    except Exception as e:
        print(f"Could not fetch base config: {e}")
        # Minimal fallback config
        model_type = base_model.split("/")[-1].split("-")[0].lower() if base_model else "llama"
        with open(os.path.join(tmp_config_dir, "config.json"), "w") as f:
            json.dump({"model_type": model_type}, f)

    result = subprocess.run(
        [sys.executable, converter, adapter_dir,
         "--outtype", "f16",
         "--outfile", out_file,
         "--base", tmp_config_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

    # Clean up temp config
    shutil.rmtree(tmp_config_dir, ignore_errors=True)

    if os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / (1024 * 1024)
        print(f"\nSUCCESS: LoRA GGUF adapter ready!")
        print(f"  File: {out_file}")
        print(f"  Size: {size_mb:.1f} MB")
        print(f"  Download from the Kaggle Output tab, then place in Unity:")
        print(f"    Assets/StreamingAssets/chemistry_instructor-lora.f16.gguf")
        print(f"  Load via SetLoraWeight on the LLM GameObject")
    else:
        print(f"\nConversion may have failed. Check output above.")
        print(f"Adapter PEFT files saved at {adapter_dir}. Convert locally with:")
        print(f"  python scripts/export_adapter.py {adapter_dir} --outtype f16")
else:
    print(f"\nConverter not found at {converter}")
    print(f"Adapter saved at {adapter_dir}. Convert locally with:")
    print(f"  python scripts/export_adapter.py {adapter_dir} --outtype f16")

In [ ]:
import sys, os
TORCH_PREFIX = '/kaggle/working/torch_env'
if TORCH_PREFIX not in sys.path: sys.path.insert(0, TORCH_PREFIX)

import os

# List all files in /kaggle/working/ so you can see what's available for download.
print("Files available in /kaggle/working/ (Output tab):")
for root, dirs, files in os.walk("/kaggle/working"):
    # Skip hidden dirs and llama.cpp source to keep output clean
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "llama.cpp"]
    for fname in files:
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {fpath}  ({size_mb:.1f} MB)")

---
### Done!

The LoRA GGUF file is a small adapter that loads at runtime in Unity.

**Download:** Go to the **Output** tab in Kaggle and download `chemistry_instructor-lora.f16.gguf`.

**Deploy to Unity:**
1. Copy the file to `Assets/StreamingAssets/chemistry_instructor-lora.f16.gguf`
2. In LLMUnity, load it via the LLM GameObject's `SetLoraWeight` method

**Resources:**
- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [Unsloth Discord](https://discord.gg/unsloth)
- [Project Repo](file:///home/athar/Projects/Unsloth_Core)